# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [ ]:
# cleaning data the same as in previous lab

# dropping rows with missing values
spaceship = spaceship.dropna()

# transforming "Cabin" to fix granualrity
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]

# Dropping unnecessary columns
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

**Perform Train Test Split**

In [8]:
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [9]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

categorical_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]
numerical_cols = X_train.select_dtypes(include="number").columns

X_train_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

train_categorical = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

test_categorical = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test.index
)

X_train_ready = pd.concat([X_train[numerical_cols], train_categorical], axis=1)
X_test_ready = pd.concat([X_test[numerical_cols], test_categorical], axis=1)

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_ready[numerical_cols] = scaler.fit_transform(X_train_ready[numerical_cols])
X_test_ready[numerical_cols] = scaler.transform(X_test_ready[numerical_cols])

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [17]:
# BAGGING

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True, # bagging
    random_state=8,
)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.7844175491679274

In [18]:
# PASTING

model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=False, # pasting
    random_state=8,
)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.7443267776096822

- Random Forests

In [20]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=8
)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

accuracy_score(y_test, y_pred)

0.7776096822995462

- Gradient Boosting

In [21]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=100,
    random_state=8
)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

accuracy_score(y_test, y_pred)

0.7904689863842662

- Adaptive Boosting

In [22]:
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(
    n_estimators=100,
    random_state=8
)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

accuracy_score(y_test, y_pred)

0.7806354009077155

Which model is the best and why?

In [24]:
# Gradient Boosting is the best model with 79.04% accuracy.
# It performs slightly better than the other models because 
# each new tree learns from the errors of previous trees.